# 01 — Ingesta ODS → SQLite

Lee `in/sheet.ods` y produce:
- `out/caracterizaciones.db` — pestaña Caracterizaciones (encuesta Google Form)
- `out/comunidades.db` — pestañas por comunidad unificadas (datos Kobo + seguimiento)

El script es idempotente: re-ejecutar reemplaza las tablas con los datos actuales del ODS.

In [ ]:
# --- Parámetros (editables via nbclick o papermill) ---
INPUT_ODS = None  # Si es None, se auto-detecta el primer *.ods en in/
OUTPUT_CARACTERIZACIONES = "out/caracterizaciones.db"
OUTPUT_COMUNIDADES = "out/comunidades.db"

TAB_CARACTERIZACIONES = "Caracterizaciones"
TABS_COMUNIDADES = [
    "Granizal",
    "La Nueva Jerusalén",
    "La Honda",
]

# Columnas que actúan como JOIN key entre tablas (nombres originales del ODS, case-insensitive)
# Caracterizaciones usa "Documento de identificación del interesado", comunidades usan "CEDULA"
JOIN_KEY_CARACTERIZACIONES = "Documento de identificación del interesado"
JOIN_KEY_COMUNIDADES = "CEDULA"

In [ ]:
import sqlite3
import logging
import sys
from pathlib import Path

from odf.opendocument import load
from odf.table import Table, TableRow, TableCell
from odf import teletype

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger("ingesta")


def find_project_root(marker="pyproject.toml"):
    """Walk up from cwd until the project root marker is found.
    Works whether the kernel was started from the project root,
    from notebooks/, or anywhere else in the tree.
    """
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / marker).exists():
            return parent
    raise RuntimeError(
        f"No se encontró la raíz del proyecto (buscando '{marker}' desde {Path.cwd()})"
    )


PROJECT_ROOT = find_project_root()
log.info(f"Project root: {PROJECT_ROOT}")

# Resolve all paths relative to project root
_in_dir  = PROJECT_ROOT / "in"
_out_dir = PROJECT_ROOT / "out"

# Resolve INPUT_ODS parameter (None = auto-detect)
if INPUT_ODS is None:
    candidates = sorted(_in_dir.glob("*.ods"))
    if not candidates:
        raise FileNotFoundError(
            f"No se encontró ningún archivo *.ods en {_in_dir}.\n"
            "Copiá la planilla exportada desde Google Sheets a la carpeta in/."
        )
    ods_path = candidates[0]
    if len(candidates) > 1:
        log.warning(f"Varios ODS en in/ — usando el primero: {ods_path.name}")
        log.warning(f"  Otros: {[p.name for p in candidates[1:]]}")
else:
    ods_path = PROJECT_ROOT / INPUT_ODS if not Path(INPUT_ODS).is_absolute() else Path(INPUT_ODS)
    if not ods_path.exists():
        raise FileNotFoundError(f"Archivo ODS no encontrado: '{ods_path}'")

# Resolve output paths
out_caracterizaciones = PROJECT_ROOT / OUTPUT_CARACTERIZACIONES
out_comunidades       = PROJECT_ROOT / OUTPUT_COMUNIDADES

log.info(f"ODS:                  {ods_path.relative_to(PROJECT_ROOT)}")
log.info(f"→ caracterizaciones:  {out_caracterizaciones.relative_to(PROJECT_ROOT)}")
log.info(f"→ comunidades:        {out_comunidades.relative_to(PROJECT_ROOT)}")

In [ ]:
_ANNOTATION_QNAME = ("urn:oasis:names:tc:opendocument:xmlns:office:1.0", "annotation")
_P_QNAME = ("urn:oasis:names:tc:opendocument:xmlns:text:1.0", "p")


def _cell_value(cell):
    """
    Extract text from an ODS cell, skipping office:annotation children.
    LibreOffice embeds annotation content (note body + author + date) into
    the teletype extraction, which corrupts header cell names.
    """
    parts = []
    for child in cell.childNodes:
        if child.qname == _ANNOTATION_QNAME:
            continue  # skip notes/annotations entirely
        if child.qname == _P_QNAME:
            parts.append(teletype.extractText(child).strip())
    text = " ".join(p for p in parts if p).strip()
    return text if text else None


def read_ods_sheet(doc, sheet_name):
    """
    Read a named sheet from an ODS document.
    Returns (headers: list[str], rows: list[dict]).
    Expands repeated columns (capped at 200 to avoid ODS blank-cell inflation).
    Skips fully-empty rows.
    """
    MAX_REPEAT = 200  # ODS encodes trailing blanks with repeat=1024+; cap it

    sheets = {
        s.getAttribute("name"): s
        for s in doc.spreadsheet.getElementsByType(Table)
    }
    available = list(sheets.keys())
    if sheet_name not in sheets:
        raise ValueError(f"Pestaña '{sheet_name}' no encontrada. Disponibles: {available}")

    sheet = sheets[sheet_name]
    raw_rows = []

    for row in sheet.getElementsByType(TableRow):
        cells = []
        for cell in row.getElementsByType(TableCell):
            repeat = min(int(cell.getAttribute("numbercolumnsrepeated") or 1), MAX_REPEAT)
            val = _cell_value(cell)
            cells.extend([val] * repeat)
        while cells and cells[-1] is None:
            cells.pop()
        raw_rows.append(cells)

    while raw_rows and not any(raw_rows[-1]):
        raw_rows.pop()

    if not raw_rows:
        raise ValueError(f"La pestaña '{sheet_name}' está vacía.")

    headers = raw_rows[0]
    rows = []
    for raw in raw_rows[1:]:
        if not any(raw):
            continue
        row_dict = {}
        for i, h in enumerate(headers):
            key = h if h else f"__col_{i}"
            row_dict[key] = raw[i] if i < len(raw) else None
        rows.append(row_dict)

    return headers, rows


def _sanitize_col(name, index):
    """Turn a column header into a safe SQLite column name."""
    import re
    if not name:
        return f"col_{index}"
    safe = (
        name.strip().lower()
        .replace(" ", "_").replace("/", "_").replace("\\", "_")
        .replace("(", "").replace(")", "")
        .replace("?", "").replace("¿", "")
        .replace(":", "").replace(",", "").replace(".", "")
        .replace("-", "_")
        .replace("á", "a").replace("é", "e").replace("í", "i")
        .replace("ó", "o").replace("ú", "u").replace("ñ", "n")
    )
    safe = re.sub(r"_+", "_", safe).strip("_")
    return safe or f"col_{index}"


def write_to_sqlite(db_path, table_name, headers, rows):
    """Write rows to SQLite table, replacing it if exists (idempotent)."""
    db_path = Path(db_path)
    db_path.parent.mkdir(parents=True, exist_ok=True)

    safe_headers = [_sanitize_col(h, i) for i, h in enumerate(headers)]

    with sqlite3.connect(db_path) as conn:
        conn.execute(f'DROP TABLE IF EXISTS "{table_name}"')
        col_defs = ", ".join(f'"{c}" TEXT' for c in safe_headers)
        conn.execute(f'CREATE TABLE "{table_name}" ({col_defs})')
        placeholders = ", ".join(["?"] * len(safe_headers))
        for row in rows:
            vals = [row.get(h) for h in headers]
            conn.execute(f'INSERT INTO "{table_name}" VALUES ({placeholders})', vals)
        conn.commit()

    return len(rows)


log.info("Funciones auxiliares cargadas.")

In [ ]:
# --- Cargar ODS ---
log.info("Cargando ODS...")
doc = load(str(ods_path))
available_tabs = [s.getAttribute("name") for s in doc.spreadsheet.getElementsByType(Table)]
log.info(f"Pestañas encontradas: {available_tabs}")

In [ ]:
# --- Ingesta: Caracterizaciones ---
log.info(f"Leyendo pestaña '{TAB_CARACTERIZACIONES}'...")
car_headers, car_rows = read_ods_sheet(doc, TAB_CARACTERIZACIONES)
log.info(f"  {len(car_rows)} filas, {len(car_headers)} columnas")

n = write_to_sqlite(out_caracterizaciones, "caracterizaciones", car_headers, car_rows)
log.info(f"  ✓ {n} filas escritas en '{out_caracterizaciones.relative_to(PROJECT_ROOT)}'")

In [ ]:
# --- Ingesta: Comunidades (unificadas) ---
all_com_rows = []
com_headers = None
schema_warnings = []

for tab in TABS_COMUNIDADES:
    if tab not in available_tabs:
        log.warning(f"  Pestaña '{tab}' no encontrada en el ODS — se omite.")
        continue

    log.info(f"Leyendo pestaña '{tab}'...")
    headers, rows = read_ods_sheet(doc, tab)
    log.info(f"  {len(rows)} filas, {len(headers)} columnas")

    if com_headers is None:
        com_headers = headers
    elif set(headers) != set(com_headers):
        only_in_new = set(headers) - set(com_headers)
        only_in_prev = set(com_headers) - set(headers)
        warn = f"Esquema distinto en '{tab}': +{only_in_new} -{only_in_prev}"
        log.warning(f"  ⚠ {warn}")
        schema_warnings.append(warn)
        # Merge headers (union)
        for h in headers:
            if h not in com_headers:
                com_headers.append(h)

    # Tag each row with source community
    for row in rows:
        row["__comunidad"] = tab
    all_com_rows.extend(rows)

if schema_warnings:
    log.warning(f"\n⚠ Las pestañas de comunidad tienen esquemas distintos entre sí.")
    log.warning(f"  Esto responde parcialmente D-03 en specs/dudas.md.")

log.info(f"Total comunidades: {len(all_com_rows)} filas de {len(TABS_COMUNIDADES)} pestañas")

In [ ]:
# Escribir comunidades.db
if all_com_rows and com_headers:
    n = write_to_sqlite(out_comunidades, "comunidades", com_headers, all_com_rows)
    log.info(f"✓ {n} filas escritas en '{out_comunidades.relative_to(PROJECT_ROOT)}'")
else:
    log.error("No se encontraron filas de comunidad. Verificá los nombres de las pestañas.")

In [ ]:
# --- Análisis: JOIN key entre tablas ---
log.info("\n=== Análisis de JOIN key ===")

car_has_key = JOIN_KEY_CARACTERIZACIONES in car_headers
com_has_key = JOIN_KEY_COMUNIDADES in (com_headers or [])

log.info(f"  Caracterizaciones → '{JOIN_KEY_CARACTERIZACIONES}': {'✓ presente' if car_has_key else '✗ NO encontrada'}")
log.info(f"  Comunidades       → '{JOIN_KEY_COMUNIDADES}':        {'✓ presente' if com_has_key else '✗ NO encontrada'}")

if not car_has_key or not com_has_key:
    log.warning("  ⚠ JOIN key no encontrada en una o ambas tablas. Revisar D-03 en specs/dudas.md.")
else:
    log.info("  → JOIN posible via cédula/documento de identificación.")

In [ ]:
# --- Análisis: Filas de comunidad sin match en caracterizaciones ---
log.info("\n=== Match por cédula entre tablas ===")

if car_has_key and com_has_key:
    car_ids = {r.get(JOIN_KEY_CARACTERIZACIONES) for r in car_rows if r.get(JOIN_KEY_CARACTERIZACIONES)}
    com_ids = {r.get(JOIN_KEY_COMUNIDADES) for r in all_com_rows if r.get(JOIN_KEY_COMUNIDADES)}

    matched   = car_ids & com_ids
    only_car  = car_ids - com_ids
    only_com  = com_ids - car_ids

    log.info(f"  Familias con cédula en ambas tablas (match):         {len(matched)}")
    log.info(f"  Solo en caracterizaciones (sin registro en comunidad): {len(only_car)}")
    log.info(f"  Solo en comunidades (sin encuesta inicial):           {len(only_com)}")

    total_car_no_cedula = sum(1 for r in car_rows if not r.get(JOIN_KEY_CARACTERIZACIONES))
    total_com_no_cedula = sum(1 for r in all_com_rows if not r.get(JOIN_KEY_COMUNIDADES))
    if total_car_no_cedula:
        log.warning(f"  ⚠ {total_car_no_cedula} filas en caracterizaciones sin cédula (no participan en match)")
    if total_com_no_cedula:
        log.warning(f"  ⚠ {total_com_no_cedula} filas en comunidades sin cédula (no participan en match)")
    if only_com:
        log.warning(f"  ⚠ {len(only_com)} familias en comunidades no tienen encuesta inicial — investigar en Fase 2.")
else:
    log.warning("  Match analysis omitido: JOIN key no disponible en ambas tablas.")

In [ ]:
# --- Resumen final ---
log.info("\n=== RESUMEN DE INGESTA ===")
log.info(f"ODS leído:              {ods_path.name}")
log.info(f"caracterizaciones.db:   {len(car_rows)} filas → {out_caracterizaciones.relative_to(PROJECT_ROOT)}")
log.info(f"comunidades.db:         {len(all_com_rows)} filas → {out_comunidades.relative_to(PROJECT_ROOT)}")
if schema_warnings:
    log.warning(f"Advertencias de esquema: {len(schema_warnings)} (ver arriba)")
log.info("Ingesta completada. ✓")